With the assistance of chatgpt4-o

---

之前已經知道：
1. 專有名詞的對齊，需要：
  * 原本的句子：包含，
    * text
    * 切過後的結果
  * 翻譯後的句子：包含，
    * text
    * 切過後的結果
  * 詞對齊的對照。
2. 例如：
  * 原本的句子
    * "張三先生去了北京"
    * ['張三', '先生', '去', '了', '北京']
  * 翻譯後的句子
    * "Mr. Zhang San went to Beijing"
    *  ["Mr.", "Zhang San", "went", "to", "Beijing"]
  * 詞對齊的對照
    * {0: 0, 1: 1, 2: 3, 3: 4}
3. 透過以上的資訊，可以修正翻譯為：
  * ['Mr.', '張三', 'went', 'to', '北京']


## 相關套件的 survey

### [HanLP](https://github.com/hankcs/HanLP)

In [ ]:
!pip install hanlp[full]

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.4/232.4 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 479.7/479.7 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.2/670.2 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import hanlp
HanLP = hanlp.load(hanlp.pretrained.mtl.CLOSE_TOK_POS_NER_SRL_DEP_SDP_CON_ELECTRA_SMALL_ZH)

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
HanLP(['張三在台北101前面拍了很多照片。'])

{'tok/fine': [['張三', '在', '台北', '101', '前面', '拍', '了', '很多', '照片', '。']],
 'tok/coarse': [['張三', '在', '台北', '101', '前面', '拍', '了', '很多', '照片', '。']],
 'pos/ctb': [['NR', 'P', 'NR', 'CD', 'NN', 'VV', 'AS', 'CD', 'NN', 'PU']],
 'pos/pku': [['nr', 'p', 'ns', 'm', 'f', 'v', 'u', 'm', 'n', 'w']],
 'pos/863': [['nh', 'p', 'ns', 'm', 'nd', 'v', 'u', 'a', 'n', 'w']],
 'ner/msra': [[('張三', 'PERSON', 0, 1),
   ('台北', 'LOCATION', 2, 3),
   ('101', 'DECIMAL', 3, 4)]],
 'ner/pku': [[('張三', 'nr', 0, 1), ('台北', 'ns', 2, 3)]],
 'ner/ontonotes': [[('張三', 'PERSON', 0, 1), ('台北101', 'FAC', 2, 4)]],
 'srl': [[[('張三', 'ARG0', 0, 1),
    ('在台北101前面', 'ARGM-LOC', 1, 5),
    ('拍', 'PRED', 5, 6),
    ('很多照片', 'ARG1', 7, 9)]]],
 'dep': [[(6, 'nsubj'),
   (6, 'prep'),
   (4, 'dep'),
   (5, 'lobj'),
   (2, 'pobj'),
   (0, 'root'),
   (6, 'asp'),
   (9, 'nummod'),
   (6, 'dobj'),
   (6, 'punct')]],
 'sdp': [[[(6, 'Agt')],
   [(4, 'mPrep')],
   [(4, 'Nmod')],
   [(6, 'Datv')],
   [(4, 'mRang')],
   [(0, 'Root')],
 

在使用 HanLP 進行命名實體識別（NER）時，像 `ner/msra`、`ner/pku`、`ner/ontonotes` 這些模型會返回一個三元組（entity, label, start, end）。其中，**後面的數字**是代表實體在句子中的**位置**，具體來說：

- **start**: 實體的**開始位置**（包含該索引）。
- **end**: 實體的**結束位置**（不包含該索引，Python 中範圍通常是左閉右開）。

例如：

```python
'ner/msra': [[('張三', 'PERSON', 0, 1), ('台北', 'LOCATION', 2, 3), ('101', 'DECIMAL', 3, 4)]]
```

- `'張三'` 是一個命名實體，類型是 `PERSON`，出現在句子中從位置 `0` 開始，位置 `1` 結束（所以實際上只包含第 `0` 個字，也就是「張三」）。
- `'台北'` 是一個地名（`LOCATION`），從位置 `2` 開始，到 `3` 結束（只包含「台北」這個詞）。
- `'101'` 是一個數字（`DECIMAL`），從位置 `3` 開始，到 `4` 結束（只包含「101」這個數字）。

同理，`ner/pku` 和 `ner/ontonotes` 也依此原則返回命名實體及其在句子中的位置範圍。

----

句子是切完後的句子（`[['張三', '在', '台北', '101', '前面', '拍', '了', '很多', '照片', '。']]`）這種數據的索引

In [ ]:
test = HanLP(['張三在台北101前面拍了很多照片。'])
print(test['tok/fine'])
print(test['tok/coarse'])
print(test['ner/ontonotes'])

[['張三', '在', '台北', '101', '前面', '拍', '了', '很多', '照片', '。']]
[['張三', '在', '台北', '101', '前面', '拍', '了', '很多', '照片', '。']]
[[('張三', 'PERSON', 0, 1), ('台北101', 'FAC', 2, 4)]]


In [ ]:
ontonotes_nounlist = [ item[0] for item in test['ner/ontonotes'][0]]
pku_nounlist = [ item[0] for item in test['ner/pku'][0]]
msra_nounlist = [ item[0] for item in test['ner/msra'][0]]

print(ontonotes_nounlist)
print(pku_nounlist)
print(msra_nounlist)

['張三', '台北101']
['張三', '台北']
['張三', '台北', '101']


In [ ]:
print(tuple(ontonotes_nounlist))
print(tuple(pku_nounlist))
print(tuple(msra_nounlist))

('張三', '台北101')
('張三', '台北')
('張三', '台北', '101')


### [spaCy](https://github.com/explosion/spaCy)

中文套件可參考[這個](https://spacy.io/models/zh)

[套件區別](https://chatgpt.com/share/74bcb851-2e76-4f8d-9e7b-3f1009087045)

In [ ]:
!pip install -qU pip setuptools wheel
!pip install -qU spacy
!python -m spacy download zh_core_web_sm
!python -m spacy download zh_core_web_md
!python -m spacy download zh_core_web_lg
!python -m spacy download zh_core_web_trf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
tf-keras 2.17.0 requires tensorflow<2.18,>=2.17, but you have tensorflow 2.13.1 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
albumentations 1.4.14 requires numpy>=1.24.4, but you have numpy 1.24.3 which is incompatible.
tensorflow 2.13.1 requires typing-extensions<4.6.0,>=3.6.6, but you have typing-extensions 4.12.2 which is incompatible.
tf-keras 2.17.0 requires tensorflow<2.18,>=2.17, but you have tensorflow 2.13.1 which is incompatible.
torchaudio 2.4.0+cu121 requires torch==2.4.0, but you have torch 2.1.2 which is incompatible.
torchvision 0.19.0+cu121 requires torch==2.4.0, but you have torch 2.1.2 w

In [ ]:
import spacy
nlp = spacy.load("zh_core_web_trf")
nlp('張三在台北101前面拍了很多照片。')

張三在台北101前面拍了很多照片。

In [ ]:
doc = nlp('張三在台北101前面拍了很多照片。')
doc.ents

(張三, 台北101)

In [ ]:
tok = list(list(token.text for token in doc))
print(tok)

['張三', '在', '台北', '101', '前面', '拍', '了', '很多', '照片', '。']


In [ ]:
ent = list(list(str(ent) for ent in doc.ents))
print(ent)

['張三', '台北101']


# 30 個繁體中文句子

* Generated by Chatgpt-4o.
* Ref with [a messy dialogue](https://chatgpt.com/share/d5b671c8-432d-4465-a8bc-5255c73b3ee6).

In [ ]:
test_sentence = {
  "sentences": [
    "張三在台北101前面拍了很多照片。",
    "李四昨天在台中火車站迷路了。",
    "王五參加了鴻海公司的年度大會。",
    "華為手機在全球市場上的銷量持續增長。",
    "張三開了一間叫做「香格里拉」的餐廳。",
    "小明帶著他的iPhone去了華山藝文中心。",
    "你知道Apple的總部在哪裡嗎？",
    "昨天在星巴克遇見了老朋友林小明。",
    "上海是中國的經濟中心。",
    "Google正在開發一個新的AI模型。",
    "馬雲創辦的阿里巴巴是中國最大的電商平台之一。",
    "珠穆朗瑪峰是世界最高的山。",
    "高雄的六合夜市是美食愛好者的天堂。",
    "臺灣大學的學生來參加這次比賽。",
    "蘋果電腦的發明改變了整個科技產業。",
    "中國移動的用戶數量每年都在增加。",
    "香港的金融市場非常活躍。",
    "IBM推出了一款全新的量子計算機。",
    "Microsoft的Windows系統是全球最常用的操作系統之一。",
    "劉德華出演了很多經典的電影。",
    "台灣的玉山是當地最高的山峰。",
    "世界衛生組織在日內瓦總部召開會議。",
    "任天堂的Switch遊戲機在全球大受歡迎。",
    "中央電視台播放了關於新冠疫情的最新報導。",
    "奧林匹克運動會將在東京舉行。",
    "特斯拉的自動駕駛技術引發了廣泛關注。",
    "小紅書是一個非常受歡迎的社交平台。",
    "Facebook的隱私政策再次引發爭議。",
    "日本的富士山每年都吸引大量遊客。",
    "李小龍的武術精神影響了全世界。"
  ]
}

### [HanLP](https://github.com/hankcs/HanLP)

In [ ]:
res_dataframe = {}
src_sent = []
fin_res = []
coars_res = []
onto_res = []
pku_res = []
msra_res = []

for source_sentence in test_sentence["sentences"]:
  src_sent.append(source_sentence)

  doc = HanLP(source_sentence)
  fin = doc['tok/fine']
  coarse = doc['tok/coarse']

  fin_res.append(fin)
  coars_res.append(coarse)

  ontonotes_nounlist = tuple([item[0] for item in doc['ner/ontonotes']])
  pku_nounlist = tuple([item[0] for item in doc['ner/pku']])
  msra_nounlist = tuple([ item[0] for item in doc['ner/msra']])

  onto_res.append(ontonotes_nounlist)
  pku_res.append(pku_nounlist)
  msra_res.append(msra_nounlist)

In [ ]:
import pandas as pd

df = pd.DataFrame(list(zip(src_sent, fin_res, coars_res, onto_res, pku_res, msra_res)),
               columns =['原句', 'fine', 'coarse', 'ner_onto', 'ner_pku' , 'ner_msra'])

In [ ]:
df.head(30)

,原句,fine,coarse,ner_onto,ner_pku,ner_msra
0,張三在台北101前面拍了很多照片。,"[張三, 在, 台北, 101, 前面, 拍, 了, 很多, 照片, 。]","[張三, 在, 台北, 101, 前面, 拍, 了, 很多, 照片, 。]","(張三, 台北101)","(張三, 台北)","(張三, 台北, 101)"
1,李四昨天在台中火車站迷路了。,"[李四, 昨天, 在, 台中, 火車站, 迷路, 了, 。]","[李四, 昨天, 在, 台中, 火車站, 迷路, 了, 。]","(李四, 昨天, 台中火車站)","(李四, 台中火車站)","(李四, 昨天, 台中火車站)"
2,王五參加了鴻海公司的年度大會。,"[王五, 參加, 了, 鴻海, 公司, 的, 年度, 大會, 。]","[王五, 參加, 了, 鴻海公司, 的, 年度, 大會, 。]","(王五, 鴻海公司)","(王五, 鴻海公司)","(王五, 鴻海公司)"
3,華為手機在全球市場上的銷量持續增長。,"[華為, 手機, 在, 全球, 市場, 上, 的, 銷量, 持續, 增長, 。]","[華為, 手機, 在, 全球, 市場, 上, 的, 銷量, 持續, 增長, 。]","(華為,)",(),"(華為,)"
4,張三開了一間叫做「香格里拉」的餐廳。,"[張三, 開, 了, 一, 間, 叫做, 「, 香格里拉, 」, 的, 餐廳, 。]","[張三, 開, 了, 一, 間, 叫做, 「, 香格里拉, 」, 的, 餐廳, 。]","(張三,)","(張三,)","(張三,)"
5,小明帶著他的iPhone去了華山藝文中心。,"[小明, 帶, 著, 他, 的, iPhone, 去, 了, 華山, 藝文, 中心, 。]","[小明, 帶, 著, 他, 的, iPhone, 去, 了, 華山藝文中心, 。]","(iPhone, 華山藝文中心)","(小明, 華山藝文中心)","(小明, 華山藝文中心)"
6,你知道Apple的總部在哪裡嗎？,"[你, 知道, Apple, 的, 總部, 在, 哪裡, 嗎, ？]","[你, 知道, Apple, 的, 總部, 在, 哪裡, 嗎, ？]","(Apple,)",(),"(Apple,)"
7,昨天在星巴克遇見了老朋友林小明。,"[昨天, 在, 星巴克, 遇見, 了, 老朋友, 林小明, 。]","[昨天, 在, 星巴克, 遇見, 了, 老朋友, 林小明, 。]","(昨天, 星巴克, 林小明)","(星巴克, 林小明)","(昨天, 星巴克, 林小明)"
8,上海是中國的經濟中心。,"[上海, 是, 中國, 的, 經濟, 中心, 。]","[上海, 是, 中國, 的, 經濟, 中心, 。]","(上海, 中國)","(上海, 中國)","(上海, 中國)"
9,Google正在開發一個新的AI模型。,"[Google, 正在, 開發, 一, 個, 新, 的, AI, 模型, 。]","[Google, 正在, 開發, 一個, 新, 的, AI, 模型, 。]","(Google,)",(),"(Google,)"


### spaCy

In [ ]:
src_sent = []
tok_res = []
ent_res = []


for source_sentence in test_sentence["sentences"]:
  src_sent.append(source_sentence)

  doc = nlp(source_sentence)
  tok = list(list(token.text for token in doc))

  tok_res.append(tok)

  ent_r = tuple(list(str(ent) for ent in doc.ents))
  ent_res.append(ent_r)


In [ ]:
import pandas as pd

df = pd.DataFrame(list(zip(src_sent, tok_res, ent_res)),
               columns =['原句', '分詞' , '實體'])

df.head(30)

,原句,分詞,實體
0,張三在台北101前面拍了很多照片。,"[張三, 在, 台北, 101, 前面, 拍, 了, 很多, 照片, 。]","(張三, 台北101)"
1,李四昨天在台中火車站迷路了。,"[李四, 昨天, 在, 台中, 火車站, 迷路, 了, 。]","(李四, 昨天, 台中火車站)"
2,王五參加了鴻海公司的年度大會。,"[王五參, 加, 了, 鴻海, 公司, 的, 年度, 大會, 。]","(王五參, 鴻海公司)"
3,華為手機在全球市場上的銷量持續增長。,"[華為, 手機, 在, 全球, 市場, 上, 的, 銷量, 持續, 增長, 。]",()
4,張三開了一間叫做「香格里拉」的餐廳。,"[張, 三, 開, 了, 一, 間, 叫做, 「, 香格里拉, 」, 的, 餐廳, 。]","(張三, 一, 香格里拉)"
5,小明帶著他的iPhone去了華山藝文中心。,"[小明, 帶著, 他, 的, iPhone, 去, 了, 華山, 藝文, 中心, 。]","(華山藝文中心,)"
6,你知道Apple的總部在哪裡嗎？,"[你, 知道, Apple, 的, 總部, 在, 哪, 裡嗎, ？]","(Apple,)"
7,昨天在星巴克遇見了老朋友林小明。,"[昨天, 在, 星巴克, 遇見, 了, 老朋友, 林小明, 。]","(星巴克, 林小明)"
8,上海是中國的經濟中心。,"[上海, 是, 中國, 的, 經濟, 中心, 。]","(上海,)"
9,Google正在開發一個新的AI模型。,"[Google, 正在, 開發, 一, 個, 新, 的, AI, 模型, 。]","(Google,)"
